In [ ]:
```xml
<VSCode.Cell language="markdown">
# 📨 Message Queues & Event Streaming Guide

**Asynchronous communication patterns for decoupled, scalable systems**

This guide covers:
- Message queue architectures
- RabbitMQ for reliable messaging
- Apache Kafka for event streaming
- Dead letter queues and error handling
- Message ordering and partitioning
- Consumer groups and parallel processing
- Exactly-once semantics
- Monitoring and alerting
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Message Queue Patterns

### RabbitMQ: Task Queue
```python
import pika
import json

# Publisher
def publish_task(task_data):
    connection = pika.BlockingConnection(
        pika.ConnectionParameters('rabbitmq')
    )
    channel = connection.channel()
    
    # Declare durable queue
    channel.queue_declare(queue='tasks', durable=True)
    
    # Publish message
    channel.basic_publish(
        exchange='',
        routing_key='tasks',
        body=json.dumps(task_data),
        properties=pika.BasicProperties(
            delivery_mode=2  # Persistent
        )
    )
    
    connection.close()

# Consumer
def process_tasks():
    connection = pika.BlockingConnection(
        pika.ConnectionParameters('rabbitmq')
    )
    channel = connection.channel()
    
    channel.queue_declare(queue='tasks', durable=True)
    
    def callback(ch, method, properties, body):
        try:
            task = json.loads(body)
            print(f"Processing task: {task}")
            
            # Process task
            result = do_work(task)
            
            # Acknowledge
            ch.basic_ack(delivery_tag=method.delivery_tag)
        except Exception as e:
            # Reject and requeue
            ch.basic_nack(delivery_tag=method.delivery_tag, requeue=True)
            logger.error(f"Task failed: {e}")
    
    channel.basic_consume(queue='tasks', on_message_callback=callback)
    
    print("Waiting for tasks...")
    channel.start_consuming()
```

### Topic-Based Pub/Sub
```python
# Publisher
def publish_event(exchange: str, routing_key: str, event: dict):
    connection = pika.BlockingConnection(
        pika.ConnectionParameters('rabbitmq')
    )
    channel = connection.channel()
    
    # Declare topic exchange
    channel.exchange_declare(
        exchange=exchange,
        exchange_type='topic',
        durable=True
    )
    
    channel.basic_publish(
        exchange=exchange,
        routing_key=routing_key,
        body=json.dumps(event)
    )

# Subscriber
def subscribe_to_events(exchange: str, routing_keys: list, callback):
    connection = pika.BlockingConnection(
        pika.ConnectionParameters('rabbitmq')
    )
    channel = connection.channel()
    
    channel.exchange_declare(exchange=exchange, exchange_type='topic')
    
    # Create exclusive queue for this subscriber
    queue = channel.queue_declare(exclusive=True).method.queue
    
    # Bind to routing keys
    for routing_key in routing_keys:
        channel.queue_bind(
            exchange=exchange,
            queue=queue,
            routing_key=routing_key
        )
    
    channel.basic_consume(queue=queue, on_message_callback=callback)
    channel.start_consuming()

# Usage
# Publisher
publish_event('user_events', 'user.created', {'user_id': 123})
publish_event('user_events', 'user.deleted', {'user_id': 456})

# Subscriber listening to user.* events
subscribe_to_events('user_events', ['user.*'], callback)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Apache Kafka for Event Streaming

### Kafka Producer
```python
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers=['kafka:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    acks='all',  # Wait for all replicas
    retries=3
)

def publish_event(topic: str, key: str, event: dict):
    """Publish event to topic"""
    
    future = producer.send(topic, value=event, key=key.encode())
    
    try:
        # Wait for confirmation
        record_metadata = future.get(timeout=10)
        print(f"Sent to {record_metadata.topic} "
              f"partition {record_metadata.partition} "
              f"offset {record_metadata.offset}")
    except Exception as e:
        print(f"Error: {e}")

# Usage
publish_event('chat-messages', 'room123', {
    'user_id': 'user123',
    'message': 'Hello!',
    'timestamp': datetime.now().isoformat()
})
```

### Kafka Consumer with Group
```python
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'chat-messages',
    bootstrap_servers=['kafka:9092'],
    group_id='analytics-group',
    auto_offset_reset='earliest',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    enable_auto_commit=False  # Manual commit
)

def process_messages():
    """Consume and process messages"""
    
    for message in consumer:
        try:
            event = message.value
            print(f"Processing: {event}")
            
            # Process event (e.g., save to analytics DB)
            save_to_analytics(event)
            
            # Commit offset
            consumer.commit()
        except Exception as e:
            logger.error(f"Error processing message: {e}")
            # Don't commit, will retry

process_messages()
```

### Partitioning & Ordering
```python
# Messages with same key go to same partition (ordering preserved)
for i in range(100):
    producer.send(
        'events',
        value={'data': i},
        key=b'user123'  # Same key → same partition
    )

# Without key (round-robin to all partitions)
for i in range(100):
    producer.send('events', value={'data': i})
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 3. Dead Letter Queues & Error Handling

### RabbitMQ Dead Letter Queue
```python
def setup_dlq():
    """Setup main queue with DLQ"""
    connection = pika.BlockingConnection(
        pika.ConnectionParameters('rabbitmq')
    )
    channel = connection.channel()
    
    # Declare DLQ
    channel.queue_declare(queue='tasks.dlq', durable=True)
    
    # Declare main queue with DLQ binding
    channel.queue_declare(
        queue='tasks',
        durable=True,
        arguments={
            'x-dead-letter-exchange': '',
            'x-dead-letter-routing-key': 'tasks.dlq',
            'x-message-ttl': 60000  # 60 seconds to live
        }
    )
    
    connection.close()

def monitor_dlq():
    """Process failed messages"""
    connection = pika.BlockingConnection(
        pika.ConnectionParameters('rabbitmq')
    )
    channel = connection.channel()
    
    def dlq_callback(ch, method, properties, body):
        message = json.loads(body)
        logger.error(f"DLQ message: {message}")
        
        # Decide: retry, alert, or discard
        retry_count = properties.headers.get('x-retry-count', 0)
        
        if retry_count < 3:
            # Retry with backoff
            new_headers = properties.headers.copy()
            new_headers['x-retry-count'] = retry_count + 1
            
            channel.basic_publish(
                exchange='',
                routing_key='tasks',
                body=body,
                properties=pika.BasicProperties(headers=new_headers)
            )
        else:
            # Give up, alert team
            alert_team(f"Message failed after 3 retries: {message}")
    
    channel.basic_consume(queue='tasks.dlq', on_message_callback=dlq_callback)
    channel.start_consuming()
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 4. Exactly-Once Semantics

```python
class IdempotentConsumer:
    def __init__(self):
        self.processed_ids = set()  # Track processed messages
    
    async def process_message(self, message: dict, message_id: str):
        """Process message idempotently"""
        
        # Check if already processed
        if message_id in self.processed_ids:
            logger.info(f"Skipping duplicate: {message_id}")
            return
        
        # Process
        await self._handle_message(message)
        
        # Mark as processed
        self.processed_ids.add(message_id)
        
        # Store in DB for persistence
        await db.execute(
            "INSERT INTO processed_messages (id) VALUES (?)",
            (message_id,)
        )

# Better: Store in database
class PersistentIdempotentConsumer:
    async def process_message(self, message: dict, message_id: str):
        """Transaction ensures exactly-once"""
        
        async with db.transaction():
            # Check if processed
            processed = await db.query_one(
                "SELECT id FROM processed WHERE id = ?",
                (message_id,)
            )
            
            if processed:
                return
            
            # Process
            await self._handle_message(message)
            
            # Mark as processed (atomic)
            await db.execute(
                "INSERT INTO processed (id) VALUES (?)",
                (message_id,)
            )
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 5. Monitoring & Alerting

```python
# Track consumer lag
def monitor_consumer_lag():
    """Alert when consumer falls behind"""
    
    while True:
        for partition in consumer.partitions():
            # Get current offset
            current = consumer.position(partition)
            
            # Get end offset
            consumer.seek_to_end(partition)
            end = consumer.position(partition)
            
            lag = end - current
            
            # Alert if lag > threshold
            if lag > 10000:
                alert(f"High lag on partition {partition}: {lag} messages")
        
        time.sleep(60)

# Message queue metrics
METRICS = {
    'messages_published': Counter(),
    'messages_consumed': Counter(),
    'processing_time': Histogram(),
    'errors': Counter(),
    'queue_depth': Gauge()
}
```
</MySCode.Cell>
```